In [1]:
import sys
sys.path.append("..")
from src.scorer import TokenScorer, load_scorer
 
import torch
import torch.nn as nn
from transformers import DistilBertModel
from src.utils import get_distilbert_tokenizer
 
tok = get_distilbert_tokenizer()

In [2]:
class TokenScorer(nn.Module):
    def __init__(self, hidden_dim=768, freeze_encoder=True):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained("distilbert-base-uncased")
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state          # [B, seq, 768]
        scores = torch.sigmoid(self.head(h)).squeeze(-1)  # [B, seq]
        return scores

In [3]:
from transformers import DistilBertTokenizerFast
tok = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = TokenScorer()

enc = tok("Hello world this is a test", return_tensors="pt")
scores = model(enc.input_ids, enc.attention_mask)
print(scores.shape)   # torch.Size([1, 8])
print(scores)         # values between 0 and 1

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 8])
tensor([[0.6901, 0.4908, 0.4795, 0.4740, 0.4518, 0.4836, 0.5632, 0.5318]],
       grad_fn=<SqueezeBackward1>)


In [4]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}  /  Total: {total:,}")

Trainable: 769  /  Total: 66,363,649
